In [2]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict, Annotated,Literal
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
import os


load_dotenv()


# llm setup
llm = HuggingFaceEndpoint(
    # repo_id="moonshotai/Kimi-K2-Instruct-0905",
    # repo_id="deepcogito/cogito-671b-v2.1-FP8",
    repo_id="zai-org/GLM-4.7-Flash",
    task="text-generation",
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN"),
)


model_hf= ChatHuggingFace(llm=llm)


# defining state
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str
    
    
def generate_joke(state:JokeState):
    print("Generating joke...")
    topic=state['topic']
    prompt=f"Generate a funny joke about {topic}."
    response = model_hf.invoke([HumanMessage(content=prompt)])
    state['joke']=response.content
    return {'joke':response.content}


def generate_explanation(state:JokeState):
    print("Generating explanation...")
    joke=state['joke']
    prompt=f"Explain the following joke in simple terms:\n{joke}"
    response = model_hf.invoke([HumanMessage(content=prompt)])
    state['explanation']=response.content
    return {'explanation':response.content}



In [3]:

# define the graph
graph=StateGraph(JokeState)

# adding nodes
graph.add_node('generate_joke',generate_joke,description="Generate a joke based on the given topic")
graph.add_node('generate_explanation',generate_explanation,description="Generate an explanation for the given joke")

# adding edges
graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)


# for memory persistence
checkpointer=InMemorySaver()

workflow=graph.compile(checkpointer=checkpointer)



In [4]:

config_1={'configurable':{'thread_id':'joke_thread_1'}}
print(workflow.invoke({'topic':'programming'},config=config_1))


Generating joke...
Generating explanation...
{'topic': 'programming', 'joke': 'Why do programmers prefer dark mode?\n\nBecause light attracts bugs.', 'explanation': 'Here is the explanation broken down:\n\n1.  **The Setup:** You know how moths are attracted to light? Programmers use "Dark Mode" on their screens, which means the background is black or dark.\n2.  **The Pun:** The word **"light"** has two meanings here:\n    *   The first meaning is the brightness on the computer screen.\n    *   The second meaning is the source of light that bugs (insects) are attracted to.\n3.  **The Conclusion:** Therefore, if the screen is bright (**light**), it will attract bugs (**light attracts bugs**). This is a play on words using the double meaning of "light."'}


In [5]:

# getting state
print("Persisted State:",workflow.get_state(config_1))



Persisted State: StateSnapshot(values={'topic': 'programming', 'joke': 'Why do programmers prefer dark mode?\n\nBecause light attracts bugs.', 'explanation': 'Here is the explanation broken down:\n\n1.  **The Setup:** You know how moths are attracted to light? Programmers use "Dark Mode" on their screens, which means the background is black or dark.\n2.  **The Pun:** The word **"light"** has two meanings here:\n    *   The first meaning is the brightness on the computer screen.\n    *   The second meaning is the source of light that bugs (insects) are attracted to.\n3.  **The Conclusion:** Therefore, if the screen is bright (**light**), it will attract bugs (**light attracts bugs**). This is a play on words using the double meaning of "light."'}, next=(), config={'configurable': {'thread_id': 'joke_thread_1', 'checkpoint_ns': '', 'checkpoint_id': '1f0f9aef-ffc6-6548-8002-510bd1a66684'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-01-25T05:30:46.284218+00:00

In [6]:

# listing all states
print("All Persisted States:")
for state in list(workflow.get_state_history(config_1)):
    print(state)
    print("===="*20)
    
    

All Persisted States:
StateSnapshot(values={'topic': 'programming', 'joke': 'Why do programmers prefer dark mode?\n\nBecause light attracts bugs.', 'explanation': 'Here is the explanation broken down:\n\n1.  **The Setup:** You know how moths are attracted to light? Programmers use "Dark Mode" on their screens, which means the background is black or dark.\n2.  **The Pun:** The word **"light"** has two meanings here:\n    *   The first meaning is the brightness on the computer screen.\n    *   The second meaning is the source of light that bugs (insects) are attracted to.\n3.  **The Conclusion:** Therefore, if the screen is bright (**light**), it will attract bugs (**light attracts bugs**). This is a play on words using the double meaning of "light."'}, next=(), config={'configurable': {'thread_id': 'joke_thread_1', 'checkpoint_ns': '', 'checkpoint_id': '1f0f9aef-ffc6-6548-8002-510bd1a66684'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-01-25T05:30:46.284218+

In [7]:

config_2={'configurable':{'thread_id':'joke_thread_2'}}
print(workflow.invoke({'topic':'artificial intelligence'},config=config_2))

# getting state
print("Persisted State:",workflow.get_state(config_2))
print('\n'*5)
# listing all states
print("All Persisted States:")
for state in list(workflow.get_state_history(config_2)):
    print(state)
    print("===="*20)
    
    


Generating joke...
Generating explanation...
{'topic': 'artificial intelligence', 'joke': 'Why did the neural network go to therapy?\n\nBecause it was suffering from **over-fitting**.', 'explanation': 'Here is the simple explanation:\n\n1.  **The Joke:** A neural network is a computer program designed to learn patterns, like a student studying for a test.\n2.  **The Pun:** "Over-fitting" is a technical computer science term that happens when a student studies **too much** for a specific practice test, memorizing the answers by heart instead of understanding the general rules.\n3.  **The Punchline:** Instead of memorizing the answers, the network memorized the details of the "practice problems" (the training data). When it was given a new, slightly different question, it failed because it was too obsessed with the details of the old questions.\n\nIn therapy, people talk about their **over-fitting** because they can\'t stop obsessing over past events and can\'t adapt to new situations.'}

# TIME TRAVELLING

In [ ]:
workflow.get_state({"configurable":{'thread_id': 'joke_thread_1','checkpoint_id': '1f0f9aee-7543-6175-8000-f5eecee4aee6'}})

StateSnapshot(values={'topic': 'programming'}, next=('generate_joke',), config={'configurable': {'thread_id': 'joke_thread_1', 'checkpoint_id': '1f0f9aee-7543-6175-8000-f5eecee4aee6'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-01-25T05:30:04.916568+00:00', parent_config={'configurable': {'thread_id': 'joke_thread_1', 'checkpoint_ns': '', 'checkpoint_id': '1f0f9aee-753e-62ea-bfff-d449130121e8'}}, tasks=(PregelTask(id='6a581d99-5806-e678-bbaa-e4ae3273c5cc', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why do programmers prefer dark mode?\n\nBecause light attracts bugs.'}),), interrupts=())

In [ ]:
# invoking the state right after it get the topic
workflow.invoke(None,config={"configurable":{'thread_id': 'joke_thread_1','checkpoint_id': '1f0f9aee-7543-6175-8000-f5eecee4aee6'}})

Generating joke...
Generating explanation...


{'topic': 'programming',
 'joke': '**Why do programmers prefer dark mode?**\n\n**Because light attracts bugs.**',
 'explanation': 'Here is the simple explanation:\n\n**1. The "Bugs" in a Computer**\nWhen a programmer is writing code, they sometimes make mistakes. These mistakes are called **"bugs."** To fix them, the programmer has to sit down and work hard.\n\n**2. The "Bugs" in Nature**\nOutside, a **"bug"** is a small insect (like a mosquito or a fly).\n\n**3. The Pun (The Joke)**\nThe joke relies on a play on words.\n*   Programmers spend a lot of time looking at their screens, so **"light attracts bugs"** implies that bright screens are like a magnet for computer bugs (mistakes).\n*   Therefore, they use **dark mode** to keep the computer bugs away.'}

In [ ]:
print("All Persisted States:")
for state in list(workflow.get_state_history(config_1)):
    print(state)
    print("===="*20)

All Persisted States:
StateSnapshot(values={'topic': 'programming', 'joke': 'Why do programmers prefer dark mode?\n\nBecause light attracts bugs.', 'explanation': 'Here is the explanation in simple terms:\n\nThe joke relies on a double meaning of the word **"bugs."**\n\n1.  **The Computer Meaning:** In programming, a "bug" is an error in the code that causes the software to malfunction.\n2.  **The Real-Life Meaning:** In nature, a bug is an insect (like a spider or ant).\n\n**The Punchline:**\nWhen a programmer uses **light mode** (a white background), they say it "attracts bugs." This is a pun (a play on words). It implies that using a white screen makes the computer more likely to have errors, just like a light turns on and attracts real bugs.\n\nUsing **dark mode** (a black background) keeps the "bugs" away, meaning it prevents the software from having errors.'}, next=(), config={'configurable': {'thread_id': 'joke_thread_1', 'checkpoint_ns': '', 'checkpoint_id': '1f0f9afc-24e7-6f43

# updating states

In [16]:
workflow.update_state({"configurable":{'thread_id': 'joke_thread_1','checkpoint_id': '1f0f9aee-7543-6175-8000-f5eecee4aee6',"checkpoint_ns":""}},{'topic':'computers'})


{'configurable': {'thread_id': 'joke_thread_1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f0f9b06-e77e-66aa-8001-f17be7b7cc0c'}}

In [ ]:
print("All Persisted States:")
for state in list(workflow.get_state_history(config_1)):
    print(state)
    print("===="*20)

All Persisted States:
StateSnapshot(values={'topic': 'computers'}, next=('generate_joke',), config={'configurable': {'thread_id': 'joke_thread_1', 'checkpoint_ns': '', 'checkpoint_id': '1f0f9b06-e77e-66aa-8001-f17be7b7cc0c'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-01-25T05:41:01.139729+00:00', parent_config={'configurable': {'thread_id': 'joke_thread_1', 'checkpoint_ns': '', 'checkpoint_id': '1f0f9aee-7543-6175-8000-f5eecee4aee6'}}, tasks=(PregelTask(id='b5344946-9692-9af3-6136-1be8f46bd617', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=())
StateSnapshot(values={'topic': 'programming', 'joke': '**Why do programmers prefer dark mode?**\n\n**Because light attracts bugs.**', 'explanation': 'Here is the simple explanation:\n\n**1. The "Bugs" in a Computer**\nWhen a programmer is writing code, they sometimes make mistakes. These mistakes are called **"bugs."** To fix them,

In [18]:
workflow.invoke(None,config={"configurable":{'thread_id': 'joke_thread_1','checkpoint_id': '1f0f9b06-e77e-66aa-8001-f17be7b7cc0c'}})


Generating joke...
Generating explanation...


{'topic': 'computers',
 'joke': 'I told my wife she was drawing her eyebrows too high.\n\nShe looked surprised.',
 'explanation': "Here is the simple explanation:\n\n**The Joke:**\nWhen you draw your eyebrows, you usually draw them in the natural shape of your face. If you draw them **too high**, it looks like your forehead is expanding.\n\n**The Punchline:**\nWhen she looked surprised, she didn't realize you were making a joke about her drawing skills. Instead, she thought she was seeing a ghost or a spirit, because your eyebrows were so high up that she looked like a **ghost**.\n\nSo, the surprise was real because she didn't realize she looked like a ghost until you told her."}

In [19]:
print("All Persisted States:")
for state in list(workflow.get_state_history(config_1)):
    print(state)
    print("===="*20)

All Persisted States:
StateSnapshot(values={'topic': 'computers', 'joke': 'I told my wife she was drawing her eyebrows too high.\n\nShe looked surprised.', 'explanation': "Here is the simple explanation:\n\n**The Joke:**\nWhen you draw your eyebrows, you usually draw them in the natural shape of your face. If you draw them **too high**, it looks like your forehead is expanding.\n\n**The Punchline:**\nWhen she looked surprised, she didn't realize you were making a joke about her drawing skills. Instead, she thought she was seeing a ghost or a spirit, because your eyebrows were so high up that she looked like a **ghost**.\n\nSo, the surprise was real because she didn't realize she looked like a ghost until you told her."}, next=(), config={'configurable': {'thread_id': 'joke_thread_1', 'checkpoint_ns': '', 'checkpoint_id': '1f0f9b09-c366-69d1-8003-c9d82bbad3c4'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-01-25T05:42:17.885742+00:00', parent_config={'configu